# ViT Training Notebook

In [5]:
import csv
import json
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms

try:
    import pandas as pd
except ImportError:
    pd = None


PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'data').exists() and (candidate / 'Model').exists()
    ),
    Path.cwd(),
)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_age_from_name(path: Path) -> int:
    stem = path.stem
    if '_' not in stem:
        raise ValueError(f"Filename does not contain '_' age separator: {path.name}")
    age_str = stem.rsplit('_', 1)[1]
    if not age_str.isdigit():
        raise ValueError(f"Age is not numeric in filename: {path.name}")
    return int(age_str)


def build_samples(root: Path) -> List[Tuple[Path, int]]:
    samples = []
    for path in root.rglob('*'):
        if path.is_file() and path.suffix.lower() in IMG_EXTS:
            age = parse_age_from_name(path)
            samples.append((path, age))
    if not samples:
        raise RuntimeError(f'No images found under {root}')
    return samples


def split_indices_by_label(samples, val_ratio: float, seed: int, adult_age: int = 21):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for idx, (_, age) in enumerate(samples):
        label = 1 if age >= adult_age else 0
        by_label[label].append(idx)

    train_idx, val_idx = [], []
    for indices in by_label.values():
        rng.shuffle(indices)
        n_items = len(indices)
        n_val = int(round(n_items * val_ratio))
        if n_items >= 2:
            n_val = max(1, min(n_items - 1, n_val))
        else:
            n_val = 0
        val_idx.extend(indices[:n_val])
        train_idx.extend(indices[n_val:])

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


class FaceAdultDataset(Dataset):
    def __init__(self, samples, tfm, adult_age: int = 21):
        self.samples = samples
        self.tfm = tfm
        self.adult_age = adult_age

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index: int):
        path, age = self.samples[index]
        image = Image.open(path).convert('RGB')
        image = self.tfm(image)
        label = 1.0 if age >= self.adult_age else 0.0
        return image, torch.tensor(label, dtype=torch.float32)


class ViTBinary(nn.Module):
    def __init__(self, pretrained: bool = False):
        super().__init__()
        weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        vit = models.vit_b_16(weights=weights)
        in_features = vit.heads.head.in_features
        vit.heads = nn.Identity()
        self.backbone = vit
        self.head = nn.Linear(in_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features).squeeze(1)


def set_backbone_trainable(model: ViTBinary, trainable: bool, fine_tune_last_n_blocks: int = 0) -> None:
    for param in model.backbone.parameters():
        param.requires_grad = trainable

    for param in model.head.parameters():
        param.requires_grad = True

    if trainable and fine_tune_last_n_blocks > 0:
        for param in model.backbone.parameters():
            param.requires_grad = False

        encoder_layers = model.backbone.encoder.layers
        for block in encoder_layers[-fine_tune_last_n_blocks:]:
            for param in block.parameters():
                param.requires_grad = True

        for param in model.backbone.encoder.ln.parameters():
            param.requires_grad = True
        model.backbone.class_token.requires_grad = True


def build_optimizer(model, lr: float, weight_decay: float):
    trainable_params = [param for param in model.parameters() if param.requires_grad]
    if not trainable_params:
        raise RuntimeError('No trainable parameters found for optimizer setup.')
    return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)


@torch.no_grad()
def evaluate(model, loader, device, criterion, threshold: float = 0.5, use_amp: bool = False) -> Dict[str, float]:
    model.eval()
    total_loss, total = 0.0, 0
    tp = fp = tn = fn = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        batch_size = images.size(0)
        total += batch_size
        total_loss += loss.item() * batch_size

        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (2 * precision * recall) / max(precision + recall, 1e-12)
    acc = (tp + tn) / max(total, 1)

    return {
        'loss': total_loss / max(total, 1),
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


def train_one_epoch(model, loader, optimizer, device, criterion, scaler=None, use_amp: bool = False, log_interval: int = 20) -> float:
    model.train()
    total_loss, total = 0.0, 0

    num_batches = len(loader)

    for batch_idx, (images, labels) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)
        total += batch_size
        total_loss += loss.item() * batch_size

        if batch_idx == 1 or batch_idx % log_interval == 0 or batch_idx == num_batches:
            print(f'  batch {batch_idx:03d}/{num_batches:03d} | loss={loss.item():.4f}')

    return total_loss / max(total, 1)


def write_history(history: List[Dict], csv_path: Path, json_path: Path) -> None:
    if not history:
        return

    fieldnames = list(history[0].keys())
    with csv_path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(history)

    with json_path.open('w', encoding='utf-8') as file:
        json.dump(history, file, indent=2)


def run_training_stage(
    stage_name: str,
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    device,
    criterion,
    epochs: int,
    threshold: float,
    stage_history_csv_path: Path,
    stage_history_json_path: Path,
    stage_best_ckpt_path: Path,
    config_to_save: Dict,
    save_every_epoch_checkpoint: bool = False,
    epoch_ckpt_dir: Path | None = None,
    scaler=None,
    use_amp: bool = False,
    train_log_interval: int = 20,
):
    history = []
    best_f1 = -1.0

    print(f'Starting {stage_name.replace("_", " ")}...')

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, criterion, scaler=scaler, use_amp=use_amp, log_interval=train_log_interval)
        val_metrics = evaluate(model, val_loader, device, criterion, threshold=threshold, use_amp=use_amp)
        lr_now = optimizer.param_groups[0]['lr']

        epoch_record = {
            'epoch': epoch,
            'stage': stage_name,
            'lr': float(lr_now),
            'train_loss': float(train_loss),
            'val_loss': float(val_metrics['loss']),
            'val_acc': float(val_metrics['acc']),
            'val_f1': float(val_metrics['f1']),
            'val_precision': float(val_metrics['precision']),
            'val_recall': float(val_metrics['recall']),
            'tp': int(val_metrics['tp']),
            'tn': int(val_metrics['tn']),
            'fp': int(val_metrics['fp']),
            'fn': int(val_metrics['fn']),
        }
        history.append(epoch_record)
        write_history(history, stage_history_csv_path, stage_history_json_path)

        checkpoint = {
            'epoch': epoch,
            'stage': stage_name,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
            'history': history,
            'config': config_to_save,
        }

        if epoch_record['val_f1'] > best_f1:
            best_f1 = epoch_record['val_f1']
            checkpoint['best_val_f1'] = best_f1
            torch.save(checkpoint, stage_best_ckpt_path)

        if save_every_epoch_checkpoint and epoch_ckpt_dir is not None:
            torch.save(checkpoint, epoch_ckpt_dir / f'{stage_name}_epoch_{epoch:03d}.pt')

        print(
            f"[{stage_name}] Epoch {epoch:03d}/{epochs} | "
            f"lr={epoch_record['lr']:.6f} | "
            f"train_loss={epoch_record['train_loss']:.4f} | "
            f"val_loss={epoch_record['val_loss']:.4f} | "
            f"val_acc={epoch_record['val_acc']:.4f} | "
            f"val_f1={epoch_record['val_f1']:.4f} | "
            f"val_precision={epoch_record['val_precision']:.4f} | "
            f"val_recall={epoch_record['val_recall']:.4f}"
        )

        if scheduler is not None:
            scheduler.step()

    return history, best_f1


In [6]:
CONFIG = {
    'data_dir': str(PROJECT_ROOT / 'data' / 'Face_Age_Dataset'),
    'out_dir': str(PROJECT_ROOT / 'Model' / 'ViT' / 'runs' / 'vit_adult_binary'),
    'batch_size': 16,
    'initial_epochs': 10,
    'fine_tune_epochs': 10,
    'initial_lr': 1e-3,
    'fine_tune_lr': 1e-5,
    'weight_decay': 1e-4,
    'val_ratio': 0.2,
    'img_size': 224,
    'num_workers': 0,
    'seed': 42,
    'pretrained': True,
    'adult_age': 21,
    'threshold': 0.457286,
    'fine_tune_last_n_blocks': 4,
    'initial_model_output': 'best_vit_initial.pt',
    'fine_tuned_model_output': 'best_vit_finetuned.pt',
    'last_model_output': 'last_vit_model.pt',
    'save_every_epoch_checkpoint': False,
    'use_amp': True,
    'train_log_interval': 20,
}

CONFIG


{'data_dir': 'D:\\JiShou\\D-Y2S3\\BMCS2074 - Artificial Intelligence\\Assignment\\AI_Assignment\\data\\Face_Age_Dataset',
 'out_dir': 'D:\\JiShou\\D-Y2S3\\BMCS2074 - Artificial Intelligence\\Assignment\\AI_Assignment\\Model\\ViT\\runs\\vit_adult_binary',
 'batch_size': 16,
 'initial_epochs': 10,
 'fine_tune_epochs': 10,
 'initial_lr': 0.001,
 'fine_tune_lr': 1e-05,
 'weight_decay': 0.0001,
 'val_ratio': 0.2,
 'img_size': 224,
 'num_workers': 0,
 'seed': 42,
 'pretrained': True,
 'adult_age': 21,
 'threshold': 0.457286,
 'fine_tune_last_n_blocks': 4,
 'initial_model_output': 'best_vit_initial.pt',
 'fine_tuned_model_output': 'best_vit_finetuned.pt',
 'last_model_output': 'last_vit_model.pt',
 'save_every_epoch_checkpoint': False,
 'use_amp': True,
 'train_log_interval': 20}

In [7]:
seed_everything(CONFIG['seed'])

data_root = Path(CONFIG['data_dir'])
out_dir = Path(CONFIG['out_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

initial_history_csv_path = out_dir / 'initial_training_history.csv'
initial_history_json_path = out_dir / 'initial_training_history.json'
fine_tune_history_csv_path = out_dir / 'fine_tuning_history.csv'
fine_tune_history_json_path = out_dir / 'fine_tuning_history.json'
combined_history_csv_path = out_dir / 'combined_history.csv'
combined_history_json_path = out_dir / 'combined_history.json'
initial_best_ckpt_path = out_dir / CONFIG['initial_model_output']
fine_tuned_best_ckpt_path = out_dir / CONFIG['fine_tuned_model_output']
last_ckpt_path = out_dir / CONFIG['last_model_output']
config_path = out_dir / 'config.json'
epoch_ckpt_dir = out_dir / 'epoch_checkpoints'
if CONFIG['save_every_epoch_checkpoint']:
    epoch_ckpt_dir.mkdir(parents=True, exist_ok=True)

samples = build_samples(data_root)
adult_count = sum(1 for _, age in samples if age >= CONFIG['adult_age'])
nonadult_count = len(samples) - adult_count

if adult_count == 0 or nonadult_count == 0:
    raise RuntimeError('Dataset must contain BOTH adult and not-adult samples.')

train_idx, val_idx = split_indices_by_label(
    samples,
    CONFIG['val_ratio'],
    CONFIG['seed'],
    adult_age=CONFIG['adult_age'],
)

if not train_idx or not val_idx:
    raise RuntimeError('Train/val split failed. Adjust val_ratio or dataset size.')

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['img_size'], scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfm = transforms.Compose([
    transforms.Resize(int(CONFIG['img_size'] * 1.14)),
    transforms.CenterCrop(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = Subset(FaceAdultDataset(samples, train_tfm, adult_age=CONFIG['adult_age']), train_idx)
val_dataset = Subset(FaceAdultDataset(samples, val_tfm, adult_age=CONFIG['adult_age']), val_idx)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_pin_memory = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=use_pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=use_pin_memory,
)

model = ViTBinary(pretrained=CONFIG['pretrained']).to(device)
pos_weight = nonadult_count / max(adult_count, 1)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
use_amp = CONFIG['use_amp'] and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

config_to_save = {
    **CONFIG,
    'resolved_data_dir': str(data_root.resolve()),
    'train_size': len(train_idx),
    'val_size': len(val_idx),
    'adult_count': adult_count,
    'nonadult_count': nonadult_count,
    'pos_weight': float(pos_weight),
    'device': str(device),
}

with config_path.open('w', encoding='utf-8') as file:
    json.dump(config_to_save, file, indent=2)

print(f'Dataset: {data_root.resolve()}')
print(f'Total images: {len(samples)} | Train: {len(train_idx)} | Val: {len(val_idx)}')
print(f'Adult(>= {CONFIG["adult_age"]}): {adult_count} | Not adult: {nonadult_count}')
print(f'Device: {device}')
print(f'Using pos_weight={pos_weight:.4f}')


Dataset: D:\JiShou\D-Y2S3\BMCS2074 - Artificial Intelligence\Assignment\AI_Assignment\data\Face_Age_Dataset
Total images: 31780 | Train: 25424 | Val: 6356
Adult(>= 21): 15890 | Not adult: 15890
Device: cuda
Using pos_weight=1.0000


In [8]:
set_backbone_trainable(model, trainable=False)
optimizer = build_optimizer(model, lr=CONFIG['initial_lr'], weight_decay=CONFIG['weight_decay'])
initial_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['initial_epochs'])

initial_history, initial_best_f1 = run_training_stage(
    stage_name='initial_training',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=initial_scheduler,
    device=device,
    criterion=criterion,
    epochs=CONFIG['initial_epochs'],
    threshold=CONFIG['threshold'],
    stage_history_csv_path=initial_history_csv_path,
    stage_history_json_path=initial_history_json_path,
    stage_best_ckpt_path=initial_best_ckpt_path,
    config_to_save=config_to_save,
    save_every_epoch_checkpoint=CONFIG['save_every_epoch_checkpoint'],
    epoch_ckpt_dir=epoch_ckpt_dir,
    scaler=scaler,
    use_amp=use_amp,
    train_log_interval=CONFIG['train_log_interval'],
)

print('')
print('Preparing fine-tuning...')
set_backbone_trainable(
    model,
    trainable=True,
    fine_tune_last_n_blocks=CONFIG['fine_tune_last_n_blocks'],
)

optimizer = build_optimizer(model, lr=CONFIG['fine_tune_lr'], weight_decay=CONFIG['weight_decay'])
fine_tune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['fine_tune_epochs'])

fine_tune_history, fine_tune_best_f1 = run_training_stage(
    stage_name='fine_tuning',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=fine_tune_scheduler,
    device=device,
    criterion=criterion,
    epochs=CONFIG['fine_tune_epochs'],
    threshold=CONFIG['threshold'],
    stage_history_csv_path=fine_tune_history_csv_path,
    stage_history_json_path=fine_tune_history_json_path,
    stage_best_ckpt_path=fine_tuned_best_ckpt_path,
    config_to_save=config_to_save,
    save_every_epoch_checkpoint=CONFIG['save_every_epoch_checkpoint'],
    epoch_ckpt_dir=epoch_ckpt_dir,
    scaler=scaler,
    use_amp=use_amp,
    train_log_interval=CONFIG['train_log_interval'],
)

combined_history = initial_history + fine_tune_history
write_history(combined_history, combined_history_csv_path, combined_history_json_path)

best_stage = 'fine_tuning' if fine_tune_best_f1 >= initial_best_f1 else 'initial_training'
best_val_f1 = max(initial_best_f1, fine_tune_best_f1)

torch.save(
    {
        'epoch': CONFIG['initial_epochs'] + CONFIG['fine_tune_epochs'],
        'stage': 'complete',
        'best_stage': best_stage,
        'best_val_f1': best_val_f1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': fine_tune_scheduler.state_dict(),
        'history': combined_history,
        'config': config_to_save,
    },
    last_ckpt_path,
)

print('')
print(f'Initial best checkpoint: {initial_best_ckpt_path}')
print(f'Fine-tuned best checkpoint: {fine_tuned_best_ckpt_path}')
print(f'Last checkpoint: {last_ckpt_path}')
print(f'Combined history CSV: {combined_history_csv_path}')
print(f'Combined history JSON: {combined_history_json_path}')

if pd is not None:
    pd.DataFrame(combined_history)
else:
    combined_history[-5:]


Starting initial training...
  batch 001/1589 | loss=0.6702
  batch 020/1589 | loss=0.6141
  batch 040/1589 | loss=0.4212
  batch 060/1589 | loss=0.6165
  batch 080/1589 | loss=0.4396
  batch 100/1589 | loss=0.3645
  batch 120/1589 | loss=0.2401
  batch 140/1589 | loss=0.5016
  batch 160/1589 | loss=0.4267
  batch 180/1589 | loss=0.2565
  batch 200/1589 | loss=0.2730
  batch 220/1589 | loss=0.2795
  batch 240/1589 | loss=0.4485
  batch 260/1589 | loss=0.4250
  batch 280/1589 | loss=0.4919
  batch 300/1589 | loss=0.3469
  batch 320/1589 | loss=0.3089
  batch 340/1589 | loss=0.4548
  batch 360/1589 | loss=0.3517
  batch 380/1589 | loss=0.2775
  batch 400/1589 | loss=0.3743
  batch 420/1589 | loss=0.2863
  batch 440/1589 | loss=0.4105
  batch 460/1589 | loss=0.1766
  batch 480/1589 | loss=0.3894
  batch 500/1589 | loss=0.6687
  batch 520/1589 | loss=0.3090
  batch 540/1589 | loss=0.4857
  batch 560/1589 | loss=0.5041
  batch 580/1589 | loss=0.5370
  batch 600/1589 | loss=0.4836
  batch 62